## 1. Load the JADES input catalogs

This case study combines two official JADES DR3 GOODS-N data products:

- **NIRCam photometric catalog:** sky coordinates, nine-band fluxes, flux uncertainties, flags, and photometric redshifts.
- **NIRSpec spectroscopic catalog:** target identifiers, sky coordinates, spectroscopic redshifts, quality grades, and observation metadata.

The catalogs are loaded through functions in `src/catalogs.py`.  
The notebook displays the intermediate products, while the catalog-specific FITS handling remains in `src/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.dr3.catalogs import (
    load_photometry_catalog,
    load_spectroscopy_catalog,
)

photometry_catalog = load_photometry_catalog()
spectroscopy_catalog = load_spectroscopy_catalog()

input_summary = pd.DataFrame(
    {
        "catalog": [
            "NIRCam photometry",
            "NIRSpec spectroscopy",
        ],
        "rows": [
            len(photometry_catalog),
            len(spectroscopy_catalog),
        ],
        "columns": [
            photometry_catalog.shape[1],
            spectroscopy_catalog.shape[1],
        ],
    }
)

display(input_summary)

display(
    photometry_catalog[
        [
            "phot_id",
            "ra_deg",
            "dec_deg",
            "z_phot",
            "flux_f090w_njy",
            "flux_f444w_njy",
            "is_flagged_star",
        ]
    ].head()
)

display(
    spectroscopy_catalog[
        [
            "nirspec_id",
            "catalog_nircam_id",
            "ra_nircam_deg",
            "dec_nircam_deg",
            "z_spec",
            "z_spec_quality",
            "dr_problem",
            "prism_exposure_s",
        ]
    ].head()
)

,catalog,rows,columns
0,NIRCam photometry,85709,38
1,NIRSpec spectroscopy,1561,17


,phot_id,ra_deg,dec_deg,z_phot,flux_f090w_njy,flux_f444w_njy,is_flagged_star
0,1000003,189.130017,62.211833,4.34,27.828562,8.907708,False
1,1000009,189.130970,62.212145,1.47,19.460276,22.014006,False
2,1000011,189.125325,62.212237,1.51,26.788431,60.832451,False
3,1000012,189.131064,62.212308,1.07,7.478627,16.239920,False
4,1000015,189.127833,62.212449,1.52,2.317305,5.510203,False


,nirspec_id,catalog_nircam_id,ra_nircam_deg,dec_nircam_deg,z_spec,z_spec_quality,dr_problem,prism_exposure_s
0,4,-9999,NaN,NaN,NaN,E,False,6214.9
1,33,1000033,189.137063,62.213274,3.909444,A,False,6214.9
2,58,1000058,189.131020,62.213989,2.442181,A,False,9322.3
3,95,1000095,189.129203,62.215139,3.909191,A,False,9322.3
4,110,1000110,189.146379,62.215508,4.064162,A,False,9322.3


### Data & SQL？

## 2. Select one spectrum per NIRCam source

A single NIRCam source may have more than one NIRSpec observation. Repeated observations must be resolved before catalog matching so that the same galaxy does not appear multiple times with duplicated labels.

For each positive published `NIRCam_ID`, the pipeline retains one preferred spectrum using the following priority:

1. no data-reduction problem;
2. a finite spectroscopic redshift;
3. the strongest redshift-quality grade (`A` before `B`, then `C`, `D`, and `E`);
4. the longest PRISM exposure when the previous criteria are tied.

The selection algorithm is implemented in `src/matching.py`. The tables below show its effect on the spectroscopic catalog.

In [2]:
from src.dr3.matching import select_best_spectrum


best_spectra, duplicate_spectra = select_best_spectrum(
    spectroscopy_catalog
)

positive_spectra = spectroscopy_catalog.loc[
    spectroscopy_catalog["catalog_nircam_id"] > 0
].copy()

repeated_id_mask = positive_spectra.duplicated(
    subset="catalog_nircam_id",
    keep=False,
)

deduplication_summary = pd.Series(
    {
        "All NIRSpec catalog rows": len(
            spectroscopy_catalog
        ),
        "Rows with a positive published NIRCam ID": len(
            positive_spectra
        ),
        "Unique positive NIRCam IDs": positive_spectra[
            "catalog_nircam_id"
        ].nunique(),
        "NIRCam IDs with repeated spectra": positive_spectra.loc[
            repeated_id_mask,
            "catalog_nircam_id",
        ].nunique(),
        "Rows belonging to repeated-ID groups": int(
            repeated_id_mask.sum()
        ),
        "Best spectra retained": len(
            best_spectra
        ),
        "Redundant spectroscopic rows removed": (
            len(positive_spectra) - len(best_spectra)
        ),
    },
    name="value",
).to_frame()

display(deduplication_summary)


duplicate_preview = duplicate_spectra.copy()

duplicate_preview["selected_as_best"] = (
    duplicate_preview["nirspec_id"].isin(
        best_spectra["nirspec_id"]
    )
)

display(
    duplicate_preview[
        [
            "catalog_nircam_id",
            "nirspec_id",
            "z_spec",
            "z_spec_quality",
            "dr_problem",
            "prism_exposure_s",
            "selected_as_best",
        ]
    ].head(15)
)

,value
All NIRSpec catalog rows,1561
Rows with a positive published NIRCam ID,1456
Unique positive NIRCam IDs,1453
NIRCam IDs with repeated spectra,3
Rows belonging to repeated-ID groups,6
Best spectra retained,1453
Redundant spectroscopic rows removed,3


,catalog_nircam_id,nirspec_id,z_spec,z_spec_quality,dr_problem,prism_exposure_s,selected_as_best
0,1005591,3991,10.605965,A,False,24859.5,True
1,1005591,5591,10.604423,A,False,9322.3,False
2,1030668,30667,NaN,E,False,3107.4,True
3,1030668,30668,NaN,E,False,3107.4,False
4,1081942,81942,2.998952,A,False,6214.9,True
5,1081942,10083470,NaN,E,False,3107.4,False


## 3. Match spectroscopy to NIRCam photometry

The deduplicated NIRSpec records are now linked to the NIRCam photometric catalog.

The matching procedure uses two stages:

1. **Published-ID match:** use the official `NIRCam_ID`, but accept it only when the photometric and spectroscopic coordinates agree within 0.2 arcsec.
2. **Nearest-sky fallback:** if the published-ID match fails, find the nearest NIRCam source on the sky and accept it only when its separation is within 0.2 arcsec.

Each accepted NIRCam source must correspond to only one selected NIRSpec record. The audit table retains the candidate IDs, angular separations, acceptance decisions, and final match method.

In [3]:
from src.dr3.config import MATCH_RADIUS_ARCSEC
from src.dr3.matching import match_catalogs

matched_catalog, match_audit = match_catalogs(
    photometry_catalog=photometry_catalog,
    best_spectra=best_spectra,
    match_radius_arcsec=MATCH_RADIUS_ARCSEC,
)

matching_summary = pd.Series(
    {
        "Best spectra entering the match": len(
            best_spectra
        ),
        "Published IDs found in photometry": int(
            match_audit[
                "published_id_in_photometry"
            ].sum()
        ),
        "Accepted published-ID matches": int(
            (
                match_audit["match_method"]
                == "exact_id"
            ).sum()
        ),
        "Accepted nearest-sky fallbacks": int(
            (
                match_audit["match_method"]
                == "nearest_sky"
            ).sum()
        ),
        "Unmatched spectroscopic records": int(
            (
                match_audit["match_method"]
                == "unmatched"
            ).sum()
        ),
        "Final one-to-one matched sources": len(
            matched_catalog
        ),
        "Accepted matches consistent with published ID": int(
            matched_catalog["id_consistent"].sum()
        ),
    },
    name="value",
).to_frame()

display(matching_summary)


match_method_summary = (
    match_audit["match_method"]
    .value_counts(dropna=False)
    .rename_axis("match_method")
    .reset_index(name="sources")
)

match_method_summary["fraction"] = (
    match_method_summary["sources"]
    / len(match_audit)
)

display(match_method_summary)


display(
    match_audit[
        [
            "nirspec_id",
            "catalog_nircam_id",
            "published_id_in_photometry",
            "exact_id_separation_arcsec",
            "nearest_phot_id",
            "nearest_separation_arcsec",
            "nearest_id_agrees",
            "selected_phot_id",
            "selected_separation_arcsec",
            "match_method",
        ]
    ].head(10)
)

,value
Best spectra entering the match,1453
Published IDs found in photometry,1451
Accepted published-ID matches,1450
Accepted nearest-sky fallbacks,1
Unmatched spectroscopic records,2
Final one-to-one matched sources,1451
Accepted matches consistent with published ID,1450


,match_method,sources,fraction
0,exact_id,1450,0.997935
1,unmatched,2,0.001376
2,nearest_sky,1,0.000688


,nirspec_id,catalog_nircam_id,published_id_in_photometry,exact_id_separation_arcsec,nearest_phot_id,nearest_separation_arcsec,nearest_id_agrees,selected_phot_id,selected_separation_arcsec,match_method
0,33,1000033,True,0.0,1000033,0.0,True,1000033,0.0,exact_id
1,58,1000058,True,0.0,1000058,0.0,True,1000058,0.0,exact_id
2,95,1000095,True,0.0,1000095,0.0,True,1000095,0.0,exact_id
3,110,1000110,True,0.0,1000110,0.0,True,1000110,0.0,exact_id
4,113,1000113,True,0.0,1000113,0.0,True,1000113,0.0,exact_id
5,146,1000146,True,0.0,1000146,0.0,True,1000146,0.0,exact_id
6,192,1000192,True,0.0,1000192,0.0,True,1000192,0.0,exact_id
7,199,1000199,True,0.0,1000199,0.0,True,1000199,0.0,exact_id
8,265,1000265,True,0.0,1000265,0.0,True,1000265,0.0,exact_id
9,420,1000420,True,0.0,1000420,0.0,True,1000420,0.0,exact_id


## 5. Display

In [4]:
from src.dr3.config import MIN_VALID_FILTERS
from src.dr3.matching import (
    add_photometric_validity,
    build_ml_ready_catalog,
)


quality_catalog = add_photometric_validity(
    matched_catalog
)

ml_ready_catalog = build_ml_ready_catalog(
    quality_catalog
)


all_accepted_matches = pd.Series(
    True,
    index=quality_catalog.index,
)


selection_stages = [
    (
        "Accepted one-to-one matches",
        all_accepted_matches,
    ),
    (
        "Finite spectroscopic redshift",
        (
            all_accepted_matches
            & quality_catalog["z_spec"].notna()
        ),
    ),
    (
        "Secure A/B/C redshift with no reduction problem",
        quality_catalog["is_secure_spec"],
    ),
    (
        "Secure source not flagged as a star",
        (
            quality_catalog["is_secure_spec"]
            & ~quality_catalog["is_flagged_star"]
        ),
    ),
    (
        (
            "ML-ready with at least "
            f"{MIN_VALID_FILTERS} valid filters"
        ),
        quality_catalog["is_ml_ready"],
    ),
]


stage_counts = pd.Series(
    {
        stage_name: int(stage_mask.sum())
        for stage_name, stage_mask
        in selection_stages
    },
    name="sources",
)


sample_attrition = stage_counts.to_frame()

sample_attrition["removed_from_previous"] = (
    sample_attrition["sources"]
    .shift(1)
    .sub(sample_attrition["sources"])
    .fillna(0)
    .astype(int)
)

sample_attrition["fraction_of_accepted_matches"] = (
    sample_attrition["sources"]
    / len(quality_catalog)
)

display(sample_attrition)


display(
    ml_ready_catalog[
        [
            "phot_id",
            "z_phot",
            "z_spec",
            "z_spec_quality",
            "match_method",
            "separation_arcsec",
            "n_valid_filters",
        ]
    ].head(10)
)

,sources,removed_from_previous,fraction_of_accepted_matches
Accepted one-to-one matches,1451,0,1.000000
Finite spectroscopic redshift,1109,342,0.764300
Secure A/B/C redshift with no reduction problem,947,162,0.652653
Secure source not flagged as a star,942,5,0.649207
ML-ready with at least 5 valid filters,846,96,0.583046


,phot_id,z_phot,z_spec,z_spec_quality,match_method,separation_arcsec,n_valid_filters
0,1000033,4.22,3.909444,A,exact_id,0.0,9
1,1000058,2.38,2.442181,A,exact_id,0.0,9
2,1000095,4.15,3.909191,A,exact_id,0.0,9
3,1000110,4.12,4.064162,A,exact_id,0.0,9
4,1000113,0.78,5.788500,A,exact_id,0.0,9
5,1000146,3.39,3.353682,A,exact_id,0.0,9
6,1000265,0.31,2.975323,A,exact_id,0.0,9
7,1000519,0.39,1.015050,C,exact_id,0.0,9
8,1000619,10.13,9.070136,A,exact_id,0.0,9
9,1000721,2.96,2.942785,A,exact_id,0.0,9


## 6. Establish the existing photometric-redshift benchmark

In [5]:
from src.dr3.redshift_metrics import (
    add_redshift_diagnostics,
    summarize_redshift_performance,
    summarize_performance_by_redshift_bin,
)

zphot_diagnostics = add_redshift_diagnostics(
    catalog=ml_ready_catalog,
    prediction_column="z_phot",
    truth_column="z_spec",
)

zphot_summary = summarize_redshift_performance(
    zphot_diagnostics
)

zphot_by_redshift = (
    summarize_performance_by_redshift_bin(
        zphot_diagnostics
    )
)

display(zphot_summary)

display(zphot_by_redshift)

worst_zphot_outliers = (
    zphot_diagnostics
    .sort_values(
        "absolute_normalized_redshift_error",
        ascending=False,
        kind="stable",
    )
    [
        [
            "phot_id",
            "z_phot",
            "z_spec",
            "z_spec_quality",
            "n_valid_filters",
            "normalized_redshift_error",
            "absolute_normalized_redshift_error",
            "is_catastrophic_outlier",
        ]
    ]
    .head(10)
)
display(worst_zphot_outliers)

,value
sources_with_finite_prediction_and_z_spec,846.000000
normalized_bias,0.001355
normalized_median_absolute_error,0.024256
sigma_nmad,0.035510
mean_absolute_redshift_error,0.310513
catastrophic_outlier_threshold,0.150000
catastrophic_outliers,96.000000
catastrophic_outlier_fraction,0.113475


,z_spec_bin,sources,median_z_spec,normalized_bias,sigma_nmad,catastrophic_outliers,catastrophic_outlier_fraction
0,"[0.0, 1.0)",64,0.695529,-0.010812,0.058018,6,0.093750
1,"[1.0, 2.0)",170,1.632026,0.008501,0.035623,12,0.070588
2,"[2.0, 3.0)",235,2.443479,-0.005181,0.036107,45,0.191489
3,"[3.0, 4.0)",164,3.400757,0.004014,0.040419,14,0.085366
4,"[4.0, 6.0)",145,4.831261,0.005615,0.019220,15,0.103448
5,"[6.0, 8.0)",54,6.742932,-0.000420,0.009111,3,0.055556
6,"[8.0, inf)",14,8.999949,0.028887,0.045363,1,0.071429


,phot_id,z_phot,z_spec,z_spec_quality,n_valid_filters,normalized_redshift_error,absolute_normalized_redshift_error,is_catastrophic_outlier
797,1080941,4.50,1.273620,C,9,1.419050,1.419050,True
586,1033500,2.69,0.847718,A,8,0.997058,0.997058,True
813,1083030,0.31,5.115093,A,9,-0.785776,0.785776,True
680,1058711,0.47,5.220073,A,9,-0.763668,0.763668,True
4,1000113,0.78,5.788500,A,9,-0.737792,0.737792,True
131,1008592,0.28,3.470217,A,9,-0.713660,0.713660,True
230,1014285,0.86,5.451142,B,9,-0.711679,0.711679,True
141,1009377,0.04,2.575630,A,9,-0.709142,0.709142,True
436,1027376,1.08,5.898095,A,9,-0.698467,0.698467,True
26,1002845,0.65,4.413591,A,9,-0.695212,0.695212,True
